# 02 · Limpieza

Convierte cada máscara en instancias listas para entrenar: una caja y un estadio por parásito o artefacto.

Kernel: **AiScope (.venv)**. La lógica está en `aiscope.data.clean` y parte de cómo pinta GDD-app, con un pincel de `80 px / zoom`:
- **Toque o arrastre macizo**: una instancia.
- **Círculo dibujado alrededor** (un hueco grande): se rellena y es una instancia.
- **Varios círculos que se tocan** (varios huecos grandes): una instancia por hueco.
- **Garabato sobre varios parásitos** (varios huecos pequeños): una instancia por grupo teñido separado, salvo esquizontes y gametocitos.
- **Caja**: se ajusta a los píxeles teñidos (oscuros y magenta) dentro del trazo; si no hay nada fiable, se usa la caja del trazo. Nunca sale del trazo.

Salidas en `data/interim/`: `images_clean.parquet` (imágenes válidas con metadatos) y `boxes_clean.parquet` (cajas con clase). La primera ejecución recorre todas las imágenes a resolución completa; las siguientes leen la caché mientras no cambien los parámetros.

In [ ]:
import sys
try:
    import aiscope  # noqa: F401
except ModuleNotFoundError:
    raise RuntimeError(f"Kernel equivocado ({sys.executable}). Usa «AiScope (.venv)»: Kernel → Change kernel.") from None

import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from aiscope import style
from aiscope.paths import RAW_DIR, INTERIM_DIR
from aiscope.data.index import build_index, to_image_coords
from aiscope.data.clean import clean_all, PARAMS
from aiscope.data.classes import CLASSES, MASK_COLORS, NOT_PARASITE, SMEAR, SPECIES, STAGE_ES
from aiscope.data.viz import boxes_crop, grid

style.apply()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

samples, images, instances_raw = build_index(RAW_DIR, INTERIM_DIR)

## 1. Pares imagen + máscara válidos

In [ ]:
pair = images["image_file"].notna() & images["mask_file"].notna()
read_err = images["read_error"].notna() if "read_error" in images else pd.Series(False, index=images.index)
display(pd.Series({
    "carpetas vacías": int(((samples["n_image_files"] == 0) & (samples["n_mask_files"] == 0)).sum()),
    "imágenes sin máscara": int((images["image_file"].notna() & images["mask_file"].isna()).sum()),
    "máscaras sin imagen": int((images["image_file"].isna() & images["mask_file"].notna()).sum()),
    "errores de lectura": int(read_err.sum()),
}).to_frame("excluidas"))

meta = samples.assign(
    especie=samples["species"].map(SPECIES),
    preparacion=samples["blood_type"].map(SMEAR),
    sesion=samples["health_facility"] + "|" + samples["microscopist"] + "|" + samples["created_on"].str[:10],
)[["folder", "species", "especie", "blood_type", "preparacion", "sample_age", "sesion"]]
imgs = images[pair & ~read_err].merge(meta, on="folder", how="left")
reescaladas = int(((imgs["mask_w"] != imgs["img_w"]) | (imgs["mask_h"] != imgs["img_h"])).sum())
print(f"pares válidos: {len(imgs)} · máscaras reescaladas a la resolución de su imagen: {reescaladas}")

## 2. Limpieza

In [ ]:
display(pd.Series({k: str(v) for k, v in PARAMS.items()}).to_frame("parámetro"))
inst = clean_all(RAW_DIR, imgs, out_path=INTERIM_DIR / "instances_clean.parquet")
inst = inst.merge(imgs[["image_id", "especie", "preparacion"]], on="image_id", how="left")
inst["estadio"] = inst["stage"].map(STAGE_ES)
print(f"{len(inst)} instancias en {inst['image_id'].nunique()} imágenes")

img_idx = imgs.set_index("image_id", drop=False)
por_comp = dict(tuple(inst.groupby(["image_id", "comp"])))

def sin_tildes(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()

def mosaico(tiles, titulo, cols=6):
    if not tiles:
        print(f"{titulo}: sin ejemplos")
        return
    fig, ax = plt.subplots(figsize=(14, 2.45 * ((len(tiles) + cols - 1) // cols)))
    ax.imshow(grid(tiles, cols)); ax.axis("off"); ax.set_title(titulo, loc="left"); plt.show()

def tiles_componentes(claves, n, seed=0):
    claves = pd.Series(claves).sample(min(n, len(claves)), random_state=seed) if len(claves) else []
    out = []
    for key in claves:
        rows = por_comp[key]
        stroke = (rows["stroke_x0"].min(), rows["stroke_y0"].min(), rows["stroke_x1"].max(), rows["stroke_y1"].max())
        boxes = list(rows[["x0", "y0", "x1", "y1"]].itertuples(index=False, name=None))
        label = f"{sin_tildes(rows['estadio'].iloc[0])} {rows['preparacion'].iloc[0]} x{len(rows)}"
        out.append(boxes_crop(RAW_DIR, img_idx.loc[key[0]], boxes, stroke, label=label))
    return out

## 3. Tipos de trazo

Rojo: caja del trazo. Verde: cajas finales.

In [ ]:
comps = inst.groupby(["image_id", "comp"]).agg(
    estadio=("estadio", "first"), split=("split", "first"), n_holes=("n_holes", "first"), instancias=("box_i", "size")).reset_index()
comps["tipo"] = np.select(
    [comps["split"] == "circulos", comps["split"] == "garabato", comps["n_holes"] == 1],
    ["varios círculos", "garabato", "círculo relleno"], "toque o arrastre")
tabla = comps.groupby("tipo").agg(componentes=("comp", "size"), instancias=("instancias", "sum"))
tabla["instancias / componente"] = (tabla["instancias"] / tabla["componentes"]).round(2)
display(tabla)
display(pd.crosstab(comps["estadio"], comps["tipo"], margins=True))
display(comps[comps["tipo"] == "garabato"]["instancias"].value_counts().sort_index().rename("garabatos por nº de instancias").to_frame())

claves = lambda t: list(comps.loc[comps["tipo"] == t, ["image_id", "comp"]].itertuples(index=False, name=None))
mosaico(tiles_componentes(claves("varios círculos"), 12), "varios círculos que se tocan")
mosaico(tiles_componentes(claves("garabato"), 24), "garabatos (al azar)")
mosaico(tiles_componentes(claves("círculo relleno"), 6), "círculos dibujados alrededor")

### Imágenes con anotación masiva

Un garabato que da más de `MAX_INST_GARABATO` instancias tapa una zona entera de gota gruesa muy parasitada, y fuera de él quedan puntos iguales sin anotar. En esas imágenes el conteo de referencia no es fiable, así que se excluyen de train, val y test.

In [ ]:
MAX_INST_GARABATO = 10
masivos = comps[(comps["tipo"] == "garabato") & (comps["instancias"] > MAX_INST_GARABATO)]
excluidas = sorted(masivos["image_id"].unique())
print(f"imágenes excluidas por anotación masiva: {len(excluidas)}")
display(masivos.merge(imgs[["image_id", "especie", "preparacion", "folder"]], on="image_id")
        .merge(samples[["folder", "comments"]], on="folder")[["image_id", "estadio", "instancias", "especie", "preparacion", "comments"]])
mosaico(tiles_componentes(list(masivos[["image_id", "comp"]].itertuples(index=False, name=None)), 6), "garabatos masivos (excluidos)", cols=3)

Fallos que siguen apareciendo en garabatos al revisarlos a ojo: parejas de anillos muy débiles, dos trofozoítos grandes que se tocan (sus núcleos no se separan) y parejas de esquizontes o gametocitos, que a propósito no se parten. Dos puntos de cromatina a menos de unos 20 px cuentan como un parásito, que es lo correcto para un anillo con doble cromatina. Los garabatos son el 4 % de los trazos.

## 4. Cajas ajustadas a la zona teñida

In [ ]:
inst["lado_caja"] = np.maximum(inst["x1"] - inst["x0"], inst["y1"] - inst["y0"])
inst["lado_trazo"] = np.maximum(inst["stroke_x1"] - inst["stroke_x0"], inst["stroke_y1"] - inst["stroke_y0"])
inst["ratio_lado"] = inst["lado_caja"] / inst["lado_trazo"]

display((100 * pd.crosstab(inst["estadio"], inst["preparacion"], values=(inst["box_method"] == "ajustada"), aggfunc="mean"))
        .round(1).rename_axis(index="% cajas ajustadas: estadio \\ preparación"))
display(inst["tight_reason"].value_counts().rename("motivo").to_frame())
display(inst[inst["box_method"] == "ajustada"].groupby("estadio")["ratio_lado"].describe(percentiles=[.1, .5, .9]).round(2)
        .rename_axis(index="lado caja / lado trazo"))

tiles = []
for (est, prep), g in inst.sample(frac=1, random_state=0).groupby(["estadio", "preparacion"]):
    for _, r in g.head(3).iterrows():
        tiles.append(boxes_crop(RAW_DIR, img_idx.loc[r["image_id"]], [(r["x0"], r["y0"], r["x1"], r["y1"])],
                                (r["stroke_x0"], r["stroke_y0"], r["stroke_x1"], r["stroke_y1"]), label=f"{sin_tildes(est)} {prep}"))
mosaico(tiles, "cajas ajustadas: 3 al azar por estadio y preparación")

fb = inst[inst["box_method"] == "trazo"]
mosaico([boxes_crop(RAW_DIR, img_idx.loc[r["image_id"]], [], (r["stroke_x0"], r["stroke_y0"], r["stroke_x1"], r["stroke_y1"]),
                    label=f"{sin_tildes(r['estadio'])} {r['preparacion']}") for _, r in fb.sample(min(12, len(fb)), random_state=0).iterrows()],
        "sin tinción fiable: se queda la caja del trazo")

## 5. Antes y después

In [ ]:
raw = to_image_coords(instances_raw[instances_raw["image_id"].isin(imgs["image_id"])], imgs)
raw["estadio"] = raw["color"].map(lambda c: STAGE_ES.get(MASK_COLORS.get(c)))
display(pd.DataFrame({"trazos (índice)": raw["estadio"].value_counts(), "instancias limpias": inst["estadio"].value_counts()})
        .assign(diferencia=lambda d: d["instancias limpias"] - d["trazos (índice)"]))

es_par = ~inst["stage"].isin(NOT_PARASITE)
raw_par = raw["estadio"] != STAGE_ES["artefact"]
conteo = pd.DataFrame({
    "antes": raw[raw_par].groupby("image_id").size(),
    "después": inst[es_par].groupby("image_id").size(),
}).reindex(imgs["image_id"]).fillna(0).astype(int)
cambio = (conteo["después"] - conteo["antes"])
print(f"imágenes cuyo conteo de parásitos cambia: {int((cambio != 0).sum())} ({(cambio != 0).mean():.1%}) · "
      f"parásitos añadidos al separar: {int(cambio.clip(lower=0).sum())}")
display(cambio.value_counts().sort_index().rename("imágenes").rename_axis(index="cambio en el conteo").to_frame())

Tamaño en píxeles con el que el modelo vería cada objeto, comparando la caja del trazo (antes) con la caja ajustada (después), según cómo se prepare la entrada.

In [ ]:
lado_campo = inst["image_id"].map(np.maximum(img_idx["field_x1"] - img_idx["field_x0"], img_idx["field_y1"] - img_idx["field_y0"]))
filas = []
for prep in ["fina", "gruesa"]:
    m = inst["preparacion"] == prep
    for nombre, lado in [("trazo", inst["lado_trazo"]), ("ajustada", inst["lado_caja"])]:
        for entrada, esc in [("campo a 640", 640), ("campo a 1280", 1280)]:
            px = (lado * esc / lado_campo)[m]
            filas.append({"preparación": prep, "caja": nombre, "entrada": entrada, "mediana px": round(px.median(), 1),
                          "p5 px": round(px.quantile(.05), 1), "% < 8 px": round(100 * (px < 8).mean(), 1),
                          "% < 12 px": round(100 * (px < 12).mean(), 1), "% < 16 px": round(100 * (px < 16).mean(), 1)})
display(pd.DataFrame(filas).set_index(["preparación", "entrada", "caja"]))

## 6. Tablas de salida

In [ ]:
n_par = inst[es_par].groupby("image_id").size()
n_art = inst[~es_par].groupby("image_id").size()
images_clean = imgs[["image_id", "folder", "idx", "zip", "image_file", "img_w", "img_h",
                     "field_x0", "field_y0", "field_x1", "field_y1", "dhash",
                     "species", "especie", "blood_type", "preparacion", "sample_age", "sesion"]].copy()
images_clean["n_parasitos"] = images_clean["image_id"].map(n_par).fillna(0).astype(int)
images_clean["n_artefactos"] = images_clean["image_id"].map(n_art).fillna(0).astype(int)
images_clean["excluir"] = images_clean["image_id"].isin(excluidas)

boxes_clean = inst[["image_id", "stage", "x0", "y0", "x1", "y1", "box_method", "split",
                    "stroke_x0", "stroke_y0", "stroke_x1", "stroke_y1"]].rename(columns={"split": "tipo_trazo"})
boxes_clean.insert(1, "class_id", boxes_clean["stage"].map(CLASSES.index).astype(int))
boxes_clean.insert(3, "es_parasito", ~boxes_clean["stage"].isin(NOT_PARASITE))

images_clean.to_parquet(INTERIM_DIR / "images_clean.parquet", index=False)
boxes_clean.to_parquet(INTERIM_DIR / "boxes_clean.parquet", index=False)
print("clases del detector:", dict(enumerate(CLASSES)))
display(boxes_clean.head())

## 7. Resumen numérico

In [ ]:
resumen = pd.Series({
    "imágenes válidas": len(images_clean),
    "imágenes excluidas (anotación masiva)": int(images_clean["excluir"].sum()),
    "máscaras reescaladas": reescaladas,
    "cajas": len(boxes_clean),
    "parásitos": int(boxes_clean["es_parasito"].sum()),
    "artefactos": int((~boxes_clean["es_parasito"]).sum()),
    "componentes partidos (círculos)": int((comps["tipo"] == "varios círculos").sum()),
    "garabatos": int((comps["tipo"] == "garabato").sum()),
    "parásitos añadidos al separar": int(cambio.clip(lower=0).sum()),
    "% cajas ajustadas": round(100 * (boxes_clean["box_method"] == "ajustada").mean(), 1),
    "imágenes sin parásitos": int((images_clean["n_parasitos"] == 0).sum()),
})
display(resumen.to_frame("valor"))
display(boxes_clean.groupby("stage").size().reindex(CLASSES).rename("cajas por clase").to_frame())